In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("22-diamonds.csv")

In [3]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [4]:
df.drop(index=df[df.x==0].index, inplace=True)
df.drop(index=df[df.y==0].index, inplace=True)
df.drop(index=df[df.z==0].index, inplace=True)

In [5]:
df = df[df['carat'] < 4]
df = df[(df['table'] < 75) & (df['table'] > 45)]
df = df[(df['y'] < 20)]
df = df[(df['z'] < 10) & (df['z'] > 2)]

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X = df.drop('price', axis =1)
y = df.price

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=15)

In [9]:
from sklearn.preprocessing import LabelEncoder

In [11]:
encoders = {}
for col in ['cut', 'color', 'clarity']:
    encoders[col] = LabelEncoder()
    X_train[col] = encoders[col].fit_transform(X_train[col])
    X_test[col] = encoders[col].transform(X_test[col])

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
from sklearn.svm import SVR

In [16]:
svr = SVR(C=1000, gamma=0.1, kernel='rbf')

In [17]:
svr.fit(X_train_scaled, y_train)

SVR(C=1000, gamma=0.1)

In [19]:
y_pred = svr.predict(X_test_scaled)

In [20]:
from sklearn.metrics import r2_score

In [21]:
r2_score(y_test, y_pred)

0.9440109323401252

In [22]:
encoders

{'cut': LabelEncoder(), 'color': LabelEncoder(), 'clarity': LabelEncoder()}

In [23]:
scaler

StandardScaler()

In [24]:
svr

SVR(C=1000, gamma=0.1)

In [25]:
import pickle

In [26]:
with open("55-diamond_model_complete.pkl", "wb") as f:
    pickle.dump(
        {
            'model' : svr,
            'encoders' : encoders,
            'scaler' : scaler
        }
        ,f
    )

In [27]:
pd.DataFrame(X_test_scaled).to_csv("55-testdatascaled.csv", index=False)